# ECE1512 — Project B, Part 1: JPEG Artifact Restoration

Train a residual CNN and compare it with the decoded JPEG on the same held-out crops and JPEG bitstreams.

Each training attempt uses a fresh `run-*` folder, even when an earlier attempt stopped before saving a checkpoint.


## Step 0 — Setup and data

Install dependencies if needed:

```bash
python -m pip install torch numpy pillow matplotlib notebook
```

Download course images from 
https://huggingface.co/datasets/LeonYe/ADPPatches

Place the `data_1um_bicubic` image folder in the parent directory of this notebook, as in the original project, or set `PROJECT_B_DATA_ROOT`. The default `full` profile uses all prepared images; `pilot` uses a fixed real-image subset, and `demo` uses separately prepared synthetic images.

In [ ]:
from pathlib import Path
import os
import sys
import zipfile

PROJECT_DIR = Path.cwd()  # Colab example: Path("/content/jpeg_restoration_notebook")
if not (PROJECT_DIR / "dataset.py").exists():
    bundle = PROJECT_DIR / "jpeg_restoration_notebook.zip"
    if bundle.exists() and not (PROJECT_DIR / "jpeg_restoration_notebook" / "dataset.py").exists():
        with zipfile.ZipFile(bundle) as archive:
            archive.extractall(PROJECT_DIR)
    if (PROJECT_DIR / "jpeg_restoration_notebook" / "dataset.py").exists():
        PROJECT_DIR = PROJECT_DIR / "jpeg_restoration_notebook"
    else:
        raise FileNotFoundError("Extract the project ZIP and set PROJECT_DIR to its module folder.")
PROJECT_DIR = PROJECT_DIR.resolve()
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR.parent))
print("Project folder:", PROJECT_DIR)



In [ ]:
# If the project requirements are not already installed, uncomment:
# %pip install -q -r ../requirements.txt


### Optional: Google Drive (Colab only)

Uncomment this cell if you use Drive. The notebook also works in local Jupyter.


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")


In [ ]:
import copy
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from train import seed_everything

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    torch.set_num_threads(min(4, torch.get_num_threads()))
seed_everything()


## 2. Configuration

In [ ]:
MODE = os.environ.get("PROJECT_B_PROFILE", "full")
if MODE not in {"full", "pilot", "demo"}:
    raise ValueError("PROJECT_B_PROFILE must be full, pilot, or demo")
default_data = PROJECT_DIR.parent / "data_1um_bicubic"
DATA_ROOT = Path(os.environ.get("PROJECT_B_DATA_ROOT", default_data)).resolve()
if MODE == "demo" and "PROJECT_B_DATA_ROOT" not in os.environ:
    raise ValueError("Set PROJECT_B_DATA_ROOT to the synthetic demo splits")
OUTPUT_ROOT = Path(os.environ.get("PROJECT_B_RUN_DIR", PROJECT_DIR / "runs" / MODE)).resolve()
TRAIN_MODELS = True
NUM_WORKERS = 0
SEED = 42
TRAIN_QUALITIES = [10, 20, 30, 40, 50]
TEST_QUALITIES = [10, 15, 20, 25, 30]

active_data_root = DATA_ROOT
CFG = {
    "train_dir": str(active_data_root / "train"),
    "val_dir": str(active_data_root / "val"),
    "test_dir": str(active_data_root / "test"),
    "epochs": 3,
    "batch_size": 16,
    "patch_size": 256,
    "eval_patch_size": 256,
    "channels": 32,
    "depth": 5,
    "qualities": TRAIN_QUALITIES,
    "lr": 2e-4,
    "num_workers": NUM_WORKERS,
    "seed": SEED,
}
if MODE == "pilot":
    CFG.update(epochs=5, batch_size=8, patch_size=96, eval_patch_size=96,
               train_limit=512, val_limit=128, test_limit=128)
elif MODE == "demo":
    CFG.update(epochs=1, batch_size=4, patch_size=32, eval_patch_size=32,
               channels=8, depth=3)
display(pd.DataFrame([CFG]).T.rename(columns={0: "Value"}))
print("Profile:", MODE, "| Output base:", OUTPUT_ROOT)


## 3. Online JPEG degradation and dataset (guide Tasks 1–2)

Each item returns `(degraded, clean, quality)`; both tensors have shape `[3,H,W]`
and values in `[0,1]`. A clean image is cropped before JPEG encode/decode.
Training uses random crops, flips, and randomly sampled quality; validation/test
use center crops and deterministic quality assignment.

JPEG uses `subsampling=0` (4:4:4), matching the supplied implementation.
Prefer lossless source images: a previously compressed JPEG source is a proxy
target, rather than an uncompressed ground truth. Evaluation uses one center
crop per source image, with the documented upsampling of undersized images.


In [ ]:
from dataset import IMG_EXTS, jpeg_compress_pil, JPEGRestorationDataset


## 4. Load the shared clean-image splits

Use the output below for the on-disk split counts. The pilot profile samples a fixed subset of each split; the full profile uses all images.


In [ ]:
from notebook_utils import check_split_overlap

display(check_split_overlap(active_data_root))

In [ ]:
train_set = JPEGRestorationDataset(CFG["train_dir"], CFG["patch_size"],
                                   CFG["qualities"], train=True, sample_limit=CFG.get("train_limit"), sample_seed=SEED)
val_set = JPEGRestorationDataset(CFG["val_dir"], CFG["patch_size"],
                                 CFG["qualities"], train=False, sample_limit=CFG.get("val_limit"), sample_seed=SEED + 1)
degraded, clean, quality = train_set[0]
print("Tensor shapes:", tuple(degraded.shape), tuple(clean.shape), "Q =", quality)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, tensor, title in zip(axes, [clean, degraded], ["Clean crop", f"JPEG Q={quality}"]):
    ax.imshow(tensor.permute(1, 2, 0))
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
plt.show()
plt.close(fig)


## 5. Residual CNN

In [ ]:
from model import ResidualRestorationCNN, count_parameters

example_model = ResidualRestorationCNN(CFG["channels"], CFG["depth"])
print(example_model)
print(f"Trainable parameters: {count_parameters(example_model):,}")
del example_model

## 6. Spatial reconstruction loss

The CNN minimizes mean absolute error between restored and clean RGB pixels:

$$\mathcal L=\operatorname{mean}|\hat{x}-x|.$$


In [ ]:
def criterion(pred, target):
    return (pred - target).abs().mean()


## 7. IQA Metrics

Implement PSNR and SSIM from scratch, without relying on existing image-quality metric libraries such as skimage.metrics, torchmetrics, or piq.


In [ ]:
from image_quality import rgb_psnr, rgb_ssim


def psnr(pred, target, data_range=1.0):
    """Return one RGB PSNR value per image, capped at 100 dB for an exact match."""
    return rgb_psnr(pred, target, data_range=data_range, mse_floor=1e-10)


def batch_ssim(pred, target):
    """Return mean RGB SSIM for a batch of normalized images."""
    return rgb_ssim(pred, target).mean()

## 8. One training step and validation



In [ ]:
from train import make_loaders, train_one_batch


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    totals = {"input_psnr": 0.0, "restored_psnr": 0.0,
              "input_ssim": 0.0, "restored_ssim": 0.0}
    count = 0
    for degraded, clean, _ in loader:
        degraded, clean = degraded.to(device), clean.to(device)
        restored = model(degraded)
        n = clean.shape[0]
        totals["input_psnr"] += psnr(degraded, clean).sum().item()
        totals["restored_psnr"] += psnr(restored, clean).sum().item()
        totals["input_ssim"] += batch_ssim(degraded, clean).item() * n
        totals["restored_ssim"] += batch_ssim(restored, clean).item() * n
        count += n
    if not count:
        raise ValueError("Validation loader is empty.")
    return {key: value / count for key, value in totals.items()}



## 9. Executable correctness checks before training

These checks cover RGB JPEG round trips, small-image resizing, deterministic
evaluation, model shape and gradients, spatial L1 loss, analytic PSNR, and
actual parameter updates. Temporary images are discarded afterward.


In [ ]:
from checks import run_correctness_checks

run_correctness_checks(seed=SEED, device=DEVICE, metric_fns=(psnr, batch_ssim),
                    train_step=train_one_batch, criterion_fn=criterion)


## 10. Training driver

Each run resets the seed, model initialization, and loader generator.
The loop below trains with AdamW (configured learning rate, weight decay `1e-4`)
and a cosine learning-rate schedule. Training losses are weighted by batch size;
validation runs after every epoch without gradients.

`best.pt` is selected by restored validation PSNR. `last.pt`, `history.json`,
and `config.json` are also saved for evaluation and plotting. Checkpoints contain
model weights and configuration, rather than optimizer state for resuming.

To evaluate an existing checkpoint, set `PROJECT_B_RUN_DIR` to the folder containing `spatial_only/` and set `TRAIN_MODELS = False`. Evaluation writes to a separate `evaluation-*` folder.


In [ ]:
from tqdm.auto import tqdm


def train_experiment(cfg, out_dir, lambda_freq=0.0, device=None):
    device = DEVICE if device is None else torch.device(device)
    cfg = copy.deepcopy(cfg)
    if lambda_freq != 0:
        raise ValueError("This notebook uses the spatial L1 baseline only")
    if cfg["epochs"] < 1:
        raise ValueError("epochs must be at least 1")
    cfg["lambda_freq"] = float(lambda_freq)
    cfg["out_dir"] = str(out_dir)
    seed_everything(cfg["seed"])

    train_loader, val_loader = make_loaders(cfg, device)
    if len(train_loader) == 0 or len(val_loader) == 0:
        raise ValueError("Training and validation loaders must both be nonempty.")
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=False)
    (out_dir / "config.json").write_text(json.dumps(cfg, indent=2))

    model = ResidualRestorationCNN(cfg["channels"], cfg["depth"]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg["epochs"]
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    print(f"{out_dir.name}: {count_parameters(model):,} parameters; device={device}")

    history = []
    best_psnr = -float("inf")
    for epoch in range(1, cfg["epochs"] + 1):
        # validate() leaves the model in eval mode, so reset it each epoch.
        model.train()
        totals = {"train_loss": 0.0}
        count = 0
        current_lr = optimizer.param_groups[0]["lr"]
        progress = tqdm(train_loader, desc=f"{out_dir.name}: {epoch}/{cfg['epochs']}")
        for degraded, clean, _ in progress:
            loss = train_one_batch(
                model, degraded, clean, criterion, optimizer, scaler, device
            )
            n = clean.shape[0]
            totals["train_loss"] += loss * n
            count += n
            progress.set_postfix(loss=f"{loss:.4f}")

        metrics = validate(model, val_loader, device)
        metrics.update({key: value / count for key, value in totals.items()})
        metrics.update(epoch=epoch, lr=current_lr)
        history.append(metrics)
        scheduler.step()

        # Keep the format expected by evaluate.load_checkpoint().
        checkpoint = {
            "model": model.state_dict(), "args": cfg,
            "epoch": epoch, "validation": metrics,
        }
        torch.save(checkpoint, out_dir / "last.pt")
        if metrics["restored_psnr"] > best_psnr:
            best_psnr = metrics["restored_psnr"]
            torch.save(checkpoint, out_dir / "best.pt")
        (out_dir / "history.json").write_text(json.dumps(history, indent=2))
        print(
            f"Epoch {epoch}: loss={metrics['train_loss']:.4f}; "
            f"val PSNR {metrics['input_psnr']:.2f} → {metrics['restored_psnr']:.2f} dB; "
            f"SSIM={metrics['restored_ssim']:.4f}"
        )

    return pd.DataFrame(history)


In [ ]:
from tempfile import mkdtemp

EXPERIMENTS = {"CNN + L1": ("spatial_only", 0.0)}
if not TRAIN_MODELS:
    for folder, _ in EXPERIMENTS.values():
        for name in ("best.pt", "history.json"):
            path = OUTPUT_ROOT / folder / name
            if not path.is_file():
                raise FileNotFoundError(f"Missing {path}. Select a run with a checkpoint and "
                                        "history, or set TRAIN_MODELS = True.")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
active_run_root = Path(mkdtemp(prefix="run-" if TRAIN_MODELS else "evaluation-", dir=OUTPUT_ROOT))
checkpoint_root = active_run_root if TRAIN_MODELS else OUTPUT_ROOT
RESULTS_DIR = active_run_root / "results"
RESULTS_DIR.mkdir()
print("Run output:", active_run_root)

histories = {}
for label, (folder, weight) in EXPERIMENTS.items():
    out_dir = checkpoint_root / folder
    if TRAIN_MODELS:
        histories[label] = train_experiment(CFG, out_dir, weight, device=DEVICE)
    else:
        histories[label] = pd.read_json(out_dir / "history.json")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for label, history in histories.items():
    axes[0].plot(history["epoch"], history["train_loss"], marker="o", label=label)
    axes[1].plot(history["epoch"], history["restored_psnr"], marker="o", label=label)
    axes[2].plot(history["epoch"], history["restored_ssim"], marker="o", label=label)
for ax, title in zip(axes, ["Training spatial L1 loss",
                             "Validation restored PSNR (dB)", "Validation restored SSIM"]):
    ax.set(xlabel="Epoch", title=title)
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "training_curves.png", dpi=160)
plt.show()
plt.close(fig)


## 12. Load the best checkpoint and evaluate method

In [ ]:
from evaluate import load_checkpoint, evaluate_all

models, loaded_configs = {}, {}
for label, (folder, _) in EXPERIMENTS.items():
    models[label], loaded_configs[label] = load_checkpoint(checkpoint_root / folder / "best.pt", device=DEVICE)

assert loaded_configs["CNN + L1"]["lambda_freq"] == 0.0
actual_train_qualities = set(loaded_configs["CNN + L1"]["qualities"])


results = evaluate_all(models, CFG, TEST_QUALITIES, device=DEVICE,
                       train_qualities=actual_train_qualities, metric_fns=(psnr, batch_ssim))
results.to_csv(RESULTS_DIR / "metrics_long.csv", index=False)
method_order = ["JPEG", "CNN + L1"]
wide = results.pivot(index=["Q", "BPP"], columns="Method", values=["PSNR", "SSIM"])
wide = wide.swaplevel(0, 1, axis=1).reindex(columns=pd.MultiIndex.from_product(
    [method_order, ["PSNR", "SSIM"]]))
wide.to_csv(RESULTS_DIR / "metrics_table.csv")
display(wide.round(4))
print("Unseen qualities:", sorted(set(TEST_QUALITIES) - actual_train_qualities))


### PSNR/SSIM versus JPEG quality and improvement over JPEG

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for method in method_order:
    subset = results[results["Method"] == method].sort_values("Q")
    for ax, metric in zip(axes, ["PSNR", "SSIM"]):
        ax.plot(subset["Q"], subset[metric], marker="o", label=method)
for ax, metric in zip(axes, ["PSNR", "SSIM"]):
    for quality in sorted(set(TEST_QUALITIES) - actual_train_qualities):
        ax.axvline(quality, linestyle=":", color="gray", alpha=0.5)
    ax.set(xlabel="JPEG quality factor Q", ylabel=metric + (" (dB)" if metric == "PSNR" else ""),
           title=metric)
    ax.set_xticks(TEST_QUALITIES)
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "quality_curves.png", dpi=160)
plt.show()
plt.close(fig)

jpeg_reference = results[results["Method"] == "JPEG"][["Q", "PSNR", "SSIM"]]
gains = results[results["Method"] != "JPEG"].merge(jpeg_reference, on="Q", suffixes=("", "_JPEG"))
gains["PSNR gain (dB)"] = gains["PSNR"] - gains["PSNR_JPEG"]
gains["SSIM gain"] = gains["SSIM"] - gains["SSIM_JPEG"]
gains = gains[["Q", "Seen in training", "Method", "PSNR gain (dB)", "SSIM gain"]]
gains.to_csv(RESULTS_DIR / "gains_over_jpeg.csv", index=False)
display(gains.round(4))


## 13. Qualitative comparisons

Choose a test image index and quality factors. Each figure shows the clean center
crop and both methods, followed by a magnified center region. The same crop
is used by quantitative evaluation. Select more than one test image for your
report, including edges, textures, and smooth regions.


In [ ]:
from visualize import show_comparison

test_images = int(results.loc[results["Method"] == "JPEG", "Images"].iloc[0])
example_indices = sorted({0, test_images // 2, test_images - 1})
for index in example_indices:
    for quality in (10, 25, 30):
        show_comparison(models, CFG, index, quality, RESULTS_DIR, mode=MODE, device=DEVICE,
                        metric_fns=(psnr, batch_ssim))


## 14. Analyze and prepare the report

Use full-data results to address these questions:

1. How much does the CNN improve PSNR and SSIM over JPEG?
2. Does the CNN help at low Q more than high Q? Use aggregate results and examples.
3. What happens at unseen Q = 15 and 25 relative to seen qualities?
4. Where do PSNR/SSIM disagree with visible quality?

Report dataset identity and split policy, source image format, all preprocessing,
training qualities, crop sizes, architecture/parameter count, optimizer, epochs,
seed, and checkpoint-selection rule. Compare JPEG and the CNN on the same evaluation crops.

Suggested sections: Introduction, Method, Experimental Setup, Results, Analysis,
and Conclusion. Optional extensions include a tiny U-Net, L1 versus MSE,
runtime analysis, and repeated seeds.

### Outputs

The `spatial_only/` directory contains `best.pt`, `last.pt`, `history.json`, and `config.json`.
The `results/` directory contains metrics tables, gains over JPEG, parameter
counts, training/quality curves, and qualitative comparisons. The next cell
packages all outputs so they can be downloaded from Jupyter or Colab.


In [ ]:
import zipfile

archive_path = active_run_root / f"jpeg_restoration_{MODE}_results.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    source_folders = [checkpoint_root / folder for folder, _ in EXPERIMENTS.values()]
    for folder in (*source_folders, RESULTS_DIR):
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                archive.write(path, Path(folder.name) / path.relative_to(folder))
print("Results archive:", archive_path)
print("Results folder:", RESULTS_DIR.resolve())
